# Multimodal AI Grand Solution — PixelSmith Local Studio

> **Mission:** Build PixelSmith — a boutique creative studio replacing $600k/year freelancer costs with an in-house AI system running entirely on local hardware (<$5k), delivering professional-grade marketing visuals in <30 seconds per 512×512 image.
>
> **Result:** 8-second generation time, 4.1/5.0 quality score, $2,500 hardware cost, 3% unusable rate, 120 images/day throughput, full text→image + video + understanding capability.

This notebook consolidates all code examples from the 13-chapter Multimodal AI track into a single executable workflow. Each section corresponds to a chapter and demonstrates the incremental progress toward the final solution.

---

## The 13-Chapter Progression

```
Ch.1: Tensor foundations          → Can load JPEGs as tensors
Ch.2: Vision Transformers         → Can embed images (768-dim semantic vectors)
Ch.3: CLIP alignment              → Can search 10k images with text queries
Ch.4: Diffusion models            → Can generate 28×28 images (5 min/image, DDPM)
Ch.5: Schedulers                  → Can generate in 30-60s (DDIM 50 steps)
Ch.6: Latent Diffusion            → Can generate 512×512 in 20s (Stable Diffusion)
Ch.7: Guidance & Conditioning     → Can control prompt adherence (CFG scale 7.5)
Ch.8: Text-to-Image Production    → Can condition on edges (ControlNet, 15s)
Ch.9: Text-to-Video               → Can generate 16-frame video clips (AnimateDiff)
Ch.10: Multimodal LLMs            → Can auto-QA generations (LLaVA, GPT-4V)
Ch.11: Audio Generation           → Can generate speech (MMS-TTS, Kokoro-82M)
Ch.12: Generative Evaluation      → Can measure quality (HPSv2: 4.1/5.0)
Ch.13: Local Diffusion Lab        → PRODUCTION: 8s per image (SDXL-Turbo)
                                   ✅ TARGET: <30s, ≥4.0/5.0, <$5k, <5% unusable
```

**Business Impact:**
- **$600k/year savings** (eliminated freelancer costs)
- **2.5-month payback** ($125k total investment / $600k annual savings)
- **40× faster turnaround** (5 days → 3 hours for 20-image campaign)
- **8× throughput increase** (15 → 120 images/day capacity)

## Setup and Imports

Install and import all required libraries for the complete multimodal AI pipeline.

In [ ]:
# Install required packages (run once)
# !pip install torch torchvision transformers diffusers accelerate xformers pillow opencv-python matplotlib numpy

In [ ]:
import torch
import torchvision.transforms as T
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import cv2

# Set up matplotlib for better visualization
plt.rcParams['figure.facecolor'] = '#1a1a2e'
plt.rcParams['axes.facecolor'] = '#1a1a2e'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = 'white'
plt.rcParams['xtick.color'] = 'white'
plt.rcParams['ytick.color'] = 'white'

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## Chapter 1: Multimodal Foundations — Raw Signals Become Tensors

**Key Concept:** Every signal (image, audio, video) must be converted to numerical tensors before a neural network can process it.

**What it unlocked:**
- Input pipeline: Load client reference JPEGs → normalize to ImageNet stats
- Foundation for all chapters: Can't train models on raw JPEGs; must tensorize first

In [ ]:
# Create a sample image tensor (simulating a loaded JPEG)
sample_image = torch.rand(3, 224, 224)  # Random RGB image

# ImageNet normalization stats (required for pretrained models)
imagenet_mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
imagenet_std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

# Normalize tensor
normalized_image = (sample_image - imagenet_mean) / imagenet_std

print(f"Original image shape: {sample_image.shape}")
print(f"Original range: [{sample_image.min():.3f}, {sample_image.max():.3f}]")
print(f"Normalized range: [{normalized_image.min():.3f}, {normalized_image.max():.3f}]")
print("\n✅ Chapter 1 Complete: Can load and tensorize images with proper normalization")

## Chapter 2: Vision Transformers — Attention Beats Convolution at Scale

**Key Concept:** ViT splits images into 16×16 patches, projects to embeddings, and applies transformer self-attention. At scale (86M+ images), ViT beats CNNs.

**What it unlocked:**
- 768-dim image embeddings: A 224×224 photo becomes 197 patch tokens (14×14 patches + CLS token)
- Transfer learning: Pretrained ViT-B/16 generalizes to any visual task
- Foundation for CLIP: ViT is the image encoder inside CLIP

In [ ]:
from transformers import ViTModel, ViTImageProcessor

# Load pretrained Vision Transformer
vit_model = ViTModel.from_pretrained('google/vit-base-patch16-224')
vit_processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224')

# Create sample image (or load from file)
sample_pil_image = Image.new('RGB', (224, 224), color='blue')

# Process and embed
inputs = vit_processor(images=sample_pil_image, return_tensors="pt")
with torch.no_grad():
    outputs = vit_model(**inputs)
    image_embedding = outputs.last_hidden_state[:, 0, :]  # CLS token

print(f"Image embedding shape: {image_embedding.shape}")
print(f"Embedding dimensionality: {image_embedding.shape[1]}")
print(f"Number of patch tokens: {outputs.last_hidden_state.shape[1]}")
print("\n✅ Chapter 2 Complete: Can extract 768-dim semantic embeddings from images")

## Chapter 3: CLIP — Teaching Vision and Language to Speak the Same Language

**Key Concept:** CLIP trains dual encoders (ViT for images, transformer for text) on 400M image-caption pairs with contrastive loss. Result: text and images project into the same 512-dim space.

**What it unlocked:**
- Zero-shot image search: Query "modern office with natural light" → rank 10k stock photos
- Text conditioning for generation: CLIP's text encoder becomes the conditioning signal in Stable Diffusion
- Prompt engineering foundation: CLIP similarity predicts which prompts will produce desired images

In [ ]:
from transformers import CLIPProcessor, CLIPModel

# Load CLIP model
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Example: Text-image similarity
text_queries = [
    "a blue geometric shape",
    "a red geometric shape",
    "a natural landscape"
]

# Process inputs
inputs = clip_processor(
    text=text_queries,
    images=sample_pil_image,
    return_tensors="pt",
    padding=True
)

# Get embeddings and compute similarity
with torch.no_grad():
    outputs = clip_model(**inputs)
    logits_per_image = outputs.logits_per_image  # Image-text similarity scores
    probs = logits_per_image.softmax(dim=1)  # Normalize to probabilities

print("Text-Image Similarity Scores:")
for i, query in enumerate(text_queries):
    print(f"  '{query}': {probs[0][i].item():.3f}")

print("\n✅ Chapter 3 Complete: Can compute text-image alignment in shared embedding space")
print("   → This enables zero-shot image search and text-conditioned generation")

## Chapter 4: Diffusion Models — Stable Generation via Denoising

**Key Concept:** DDPM generates images by learning to reverse a noise-injection process. Forward: gradually add Gaussian noise over T=1000 steps. Reverse: train U-Net to predict the added noise at each step, then denoise backward from noise → image.

**What it unlocked:**
- Generative capability: Can create entirely new images never seen in training
- Training stability: No adversarial game like GANs → stable convergence
- Mathematical foundation: Forward process has closed-form solution

In [ ]:
# Demonstrate DDPM forward process (adding noise)
def ddpm_forward_step(x0, t, T=1000):
    """
    DDPM forward process: x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * epsilon
    
    Args:
        x0: Original image tensor
        t: Current timestep (0 to T)
        T: Total timesteps
    """
    # Linear noise schedule
    beta = torch.linspace(1e-4, 0.02, T)
    alpha = 1 - beta
    alpha_bar = torch.cumprod(alpha, dim=0)
    
    # Get noise schedule values for timestep t
    alpha_bar_t = alpha_bar[t]
    
    # Sample noise
    epsilon = torch.randn_like(x0)
    
    # Forward diffusion equation
    xt = torch.sqrt(alpha_bar_t) * x0 + torch.sqrt(1 - alpha_bar_t) * epsilon
    
    return xt, epsilon

# Demonstrate noise addition at different timesteps
timesteps_to_show = [0, 250, 500, 750, 999]
fig, axes = plt.subplots(1, len(timesteps_to_show), figsize=(15, 3))

for idx, t in enumerate(timesteps_to_show):
    noisy_img, _ = ddpm_forward_step(sample_image, t)
    axes[idx].imshow(noisy_img.permute(1, 2, 0).clamp(0, 1))
    axes[idx].set_title(f't={t}')
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

print("\n✅ Chapter 4 Complete: Understand DDPM forward process (noise injection)")
print("   → Reverse process would train U-Net to predict epsilon and denoise")
print("   → Full implementation: 5 min per 28×28 image (too slow for production)")

## Chapter 5: Schedulers — 10× Speedup Without Retraining

**Key Concept:** DDPM requires T=1000 denoising steps (5 minutes). DDIM and DPM-Solver skip steps intelligently, reducing to 50 steps (30 seconds) or even 4 steps (8 seconds with LCM) without retraining the U-Net.

**What it unlocked:**
- 30-60 second generation: DDIM 50 steps on educational proxy
- Deterministic mode: DDIM with η=0 removes stochastic term → same seed always produces same image
- Foundation for production: Schedulers are the first optimization every production system applies

In [ ]:
from diffusers import DDIMScheduler, DDPMScheduler

# Compare scheduler step counts
schedulers = {
    'DDPM (baseline)': DDPMScheduler(num_train_timesteps=1000),
    'DDIM (fast)': DDIMScheduler(num_train_timesteps=1000)
}

print("Scheduler Comparison for Production:")
print("="*70)
print(f"{'Scheduler':<20} {'Steps':<10} {'Time (est.)':<15} {'Quality'}")
print("="*70)
print(f"{'DDPM':<20} {1000:<10} {'~5 min':<15} {'Best'}")
print(f"{'DDIM':<20} {50:<10} {'~30s':<15} {'Good'}")
print(f"{'DDIM':<20} {20:<10} {'~15s':<15} {'Acceptable'}")
print(f"{'LCM (Ch.13)':<20} {4:<10} {'~8s':<15} {'Production-ready'}")
print("="*70)

print("\n✅ Chapter 5 Complete: Can reduce inference steps from 1000 → 50 (10× speedup)")
print("   → Enables 30-second generation target")
print("   → Deterministic mode (η=0) for reproducible client approvals")

## Chapter 6: Latent Diffusion — Compress, Diffuse, Decode

**Key Concept:** Pixel-space DDPM on 512×512 costs 786k dimensions per step. Stable Diffusion compresses images to 64×64×4 latents with a VAE encoder (16× smaller), diffuses there (16× cheaper), then decodes back to pixels with VAE decoder.

**What it unlocked:**
- 512×512 generation in 20 seconds: Latent compression makes high-resolution generation feasible
- CLIP text conditioning: Text embeddings condition the U-Net via cross-attention
- The Stable Diffusion architecture: VAE encoder + U-Net (latent space) + VAE decoder + CLIP text encoder

In [ ]:
from diffusers import StableDiffusionPipeline, DDIMScheduler

# Load Stable Diffusion pipeline (this will download ~5GB on first run)
print("Loading Stable Diffusion pipeline...")
print("Note: First run will download ~5GB of model weights")

# Use a smaller/faster model for demonstration
model_id = "runwayml/stable-diffusion-v1-5"

# Check if CUDA is available, otherwise use CPU (slower)
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

# Initialize pipeline
pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=dtype,
    safety_checker=None  # Disable for faster loading
)
pipe = pipe.to(device)

# Swap to DDIM scheduler for faster inference (Chapter 5 technique)
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)

print(f"✅ Pipeline loaded on {device.upper()}")
print(f"   Components: VAE Encoder → U-Net (latent space) → VAE Decoder")
print(f"   Text encoder: CLIP (Chapter 3)")
print(f"   Scheduler: DDIM (Chapter 5)")
print(f"   Latent space: 64×64×4 (16× compression vs. 512×512×3 pixels)")

## Chapter 7: Guidance & Conditioning — Controlling What Gets Generated

**Key Concept:** Classifier-free guidance (CFG) scales the conditional score to amplify prompt adherence. Formula: `ε̂ = ε̂_uncond + guidance_scale × (ε̂_cond - ε̂_uncond)`. Guidance scale 1.0 = normal, 7.5 = balanced, 12.0 = strict.

**What it unlocked:**
- Prompt control: "Modern office" without guidance → generic. With guidance 7.5 → matches brief closely
- Negative prompts: Subtract unwanted concepts ("blurry, watermark, low quality")
- Guidance scale dial: The single most important hyperparameter for quality

In [ ]:
# Generate image with classifier-free guidance
prompt = "modern office interior, natural light, minimalist design, professional photography"
negative_prompt = "blurry, low quality, watermark, text, cluttered, dark"

# VisualForge production settings
guidance_scale = 7.5  # Balanced prompt adherence
num_inference_steps = 20  # DDIM fast sampling
seed = 42  # Deterministic for client approval

print("Generating image with Classifier-Free Guidance...")
print(f"Prompt: {prompt}")
print(f"Negative: {negative_prompt}")
print(f"Guidance scale: {guidance_scale}")
print(f"Steps: {num_inference_steps}")
print(f"Seed: {seed}")

generator = torch.Generator(device=device).manual_seed(seed)

# Generate (takes ~15-20 seconds on CPU, ~5-8 seconds on GPU)
with torch.no_grad():
    result = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        guidance_scale=guidance_scale,
        num_inference_steps=num_inference_steps,
        generator=generator,
        height=512,
        width=512
    )
    generated_image = result.images[0]

# Display result
plt.figure(figsize=(8, 8))
plt.imshow(generated_image)
plt.axis('off')
plt.title(f"Generated with CFG scale={guidance_scale}")
plt.tight_layout()
plt.show()

print("\n✅ Chapter 7 Complete: Can control prompt adherence with guidance scale")
print("   → Negative prompts filter out unwanted concepts")
print("   → Deterministic generation (seed) for reproducible results")

## Chapter 8: Text-to-Image Production — ControlNet for Structural Conditioning

**Key Concept:** ControlNet adds a parallel U-Net branch that injects spatial conditioning (edges, depth maps, pose skeletons) into the main U-Net via zero-initialized residual connections.

**What it unlocked:**
- <5% unusable rate: Pure text prompts → 25% unusable. ControlNet + edge map → 3% unusable
- Client control: Designer sketches composition → ControlNet generates photo matching sketch
- Inpainting & img2img: Partial generation workflows for iterative refinement

In [ ]:
from diffusers import ControlNetModel, StableDiffusionControlNetPipeline

# Load ControlNet for edge conditioning (Canny edge detection)
print("Loading ControlNet pipeline...")
print("Note: This demonstrates the concept; full ControlNet requires additional ~1.5GB download")

# ControlNet architecture
print("\nControlNet Architecture:")
print("="*60)
print("Reference Image → Edge Detection (Canny) → Edge Map")
print("                                            ↓")
print("Text Prompt → CLIP Encoder ────────────→ U-Net (with ControlNet conditioning)")
print("                                            ↓")
print("                                      VAE Decoder")
print("                                            ↓")
print("                                  Generated 512×512 Image")
print("="*60)

# Example: Canny edge detection preprocessing
def get_canny_edges(image, low_threshold=100, high_threshold=200):
    """Extract edge map from reference image using Canny detector"""
    image_np = np.array(image)
    edges = cv2.Canny(image_np, low_threshold, high_threshold)
    edges = edges[:, :, None].repeat(3, axis=2)  # Convert to 3-channel
    return Image.fromarray(edges)

print("\n✅ Chapter 8 Complete: Understand ControlNet structural conditioning")
print("   → Edge maps guide composition (product centered, specific angles)")
print("   → Depth maps ensure correct spatial layering")
print("   → Pose skeletons control human figures in fashion briefs")
print("   → Unusable rate drops from 25% → 3% with ControlNet")

# Production workflow example
print("\nVisualForge Production Workflow with ControlNet:")
print("1. Designer sketches composition in Figma")
print("2. Export sketch as edge map (Canny detection)")
print("3. Text prompt: 'Premium leather handbag, white background, studio lighting'")
print("4. ControlNet generates 10 variations matching sketch structure")
print("5. Auto-QA filters (Chapter 10) → Human review top 3 → Client approval")
print("6. Result: 15 seconds per image, 3% unusable, perfect composition control")

## Chapter 9: Text-to-Video — Extending Diffusion to Temporal Dimension

**Key Concept:** Video diffusion extends image diffusion to 4D tensors (T×C×H×W). AnimateDiff adds temporal attention layers to frozen Stable Diffusion.

**What it unlocked:**
- 16-frame video clips: Text prompt → short video (1-2 seconds at 8fps) with temporal coherence
- Motion control: ControlNet for video → condition on optical flow or first-frame sketch
- Video understanding: MLLMs (Ch.10) can answer questions about generated videos

In [ ]:
# Video generation concept (requires specialized models)
print("Text-to-Video Progression:")
print("="*70)
print("Image Generation (Ch.6-8):    (C, H, W) = (3, 512, 512)")
print("Video Generation (Ch.9):      (T, C, H, W) = (16, 3, 512, 512)")
print("="*70)

print("\nKey Challenge: Temporal Consistency")
print("  Problem: Generating 16 independent images → severe flickering")
print("  Solution: Temporal attention layers learn inter-frame correlations")
print("  Result: Smooth motion (walking, camera pan, object rotation)")

print("\nComputational Cost:")
print(f"  Image (512×512): {512*512*3:,} values per frame")
print(f"  Video (16 frames): {512*512*3*16:,} values total (16× more compute)")

print("\n✅ Chapter 9 Complete: Understand video generation as temporal extension")
print("   → AnimateDiff: Add temporal layers to frozen Stable Diffusion")
print("   → CogVideoX: Train end-to-end on video data")
print("   → Production: 5-10 second hero clips for social media campaigns")
print("   → Aggressive quantization required for 16-frame generation on laptop")

## Chapter 10: Multimodal LLMs — When Language Models Can See

**Key Concept:** LLaVA connects a frozen ViT image encoder to a frozen LLM decoder via a learned projection layer. BLIP-2 uses a Q-Former to compress visual tokens.

**What it unlocked:**
- Auto-QA workflow: Generate image → LLaVA checks quality → auto-approve or flag for review
- 120 images/day throughput: Manual QA bottleneck removed (100% human review → 15% human review)
- Visual understanding for agents: MLLMs enable screenshot understanding, UI navigation, document analysis

In [ ]:
from transformers import LlavaForConditionalGeneration, AutoProcessor

# Load multimodal LLM for visual question answering
print("Loading Multimodal LLM (LLaVA-1.5)...")
print("Note: This requires ~13GB download on first run")
print("For demonstration, we'll show the QA workflow conceptually")

# Auto-QA workflow for VisualForge
def visualforge_auto_qa(image, brief_type):
    """
    Automated quality assurance using multimodal LLM
    
    Args:
        image: Generated PIL Image
        brief_type: Campaign type ('product-on-white', 'lifestyle', 'editorial-fashion')
    
    Returns:
        dict with qa_result, confidence, auto_approve
    """
    # QA prompt templates by brief type
    qa_prompts = {
        'product-on-white': """
            This image was generated for a product-on-white campaign.
            Answer these questions:
            1. Is the product centered in frame? (yes/no)
            2. Is the background solid white with no texture? (yes/no)
            3. Is the lighting even with no harsh shadows? (yes/no)
            4. Overall quality rating (1-5 scale):
        """,
        'lifestyle': """
            This image was generated for a lifestyle campaign.
            Answer these questions:
            1. Is the lighting natural and well-balanced? (yes/no)
            2. Is the composition visually appealing? (yes/no)
            3. Does the scene feel authentic and lived-in? (yes/no)
            4. Overall quality rating (1-5 scale):
        """,
        'editorial-fashion': """
            This image was generated for an editorial fashion campaign.
            Answer these questions:
            1. Is the model's pose editorial-style? (yes/no)
            2. Is the clothing visible and in focus? (yes/no)
            3. Is the composition striking and artistic? (yes/no)
            4. Overall quality rating (1-5 scale):
        """
    }
    
    # In production, would call LLaVA here
    # qa_result = mllm.generate(prompt=qa_prompts[brief_type], image=image)
    
    return {
        'qa_prompt': qa_prompts[brief_type],
        'auto_approve_threshold': 0.9,  # Confidence > 0.9 → auto-approve
        'human_review_threshold': 0.7   # Confidence < 0.7 → flag for review
    }

# Demonstrate QA workflow
example_qa = visualforge_auto_qa(generated_image, 'product-on-white')

print("\nVisualForge Auto-QA Workflow:")
print("="*70)
print("Step 1: Generate image (Ch.6-8)")
print("Step 2: LLaVA visual QA (Ch.10)")
print("Step 3: Decision logic:")
print(f"  - Confidence > {example_qa['auto_approve_threshold']} → Auto-approve (85% of images)")
print(f"  - Confidence < {example_qa['human_review_threshold']} → Reject, regenerate")
print(f"  - In between → Human review queue (15% of images)")
print("="*70)

print("\nImpact on Throughput:")
print(f"  Before Auto-QA: 100% human review = 85 images/day max")
print(f"  After Auto-QA:  15% human review = 120 images/day capacity")
print(f"  QA time reduction: 80% (from 10 min → 2 min per image)")

print("\n✅ Chapter 10 Complete: Automated quality assurance with multimodal LLMs")
print("   → LLaVA-7B runs on same laptop as Stable Diffusion (no additional hardware)")
print("   → Visual question answering removes manual QA bottleneck")
print("   → Adapter pattern: frozen ViT + frozen LLM + learned projection layer")

## Chapter 11: Audio Generation — Local CPU Speech Synthesis

**Key Concept:** Text-to-speech using compact pretrained models (MMS-TTS, Kokoro-82M) that run on CPU without GPU.

**What it unlocked:**
- Audio modality: PixelSmith expands from text+image+video to full multimodal
- CPU-first quick win: No GPU required → accessible demo
- Foundation for audio campaigns: Voice-over for video ads, podcast intros

In [ ]:
# Audio generation concept (requires specialized TTS models)
print("Text-to-Speech Pipeline:")
print("="*70)
print("Text Input → Phoneme Sequence → Mel Spectrogram → Vocoder → Waveform")
print("="*70)

print("\nCPU-Friendly TTS Models:")
print("  - MMS-TTS (Meta): 82M parameters, 1100+ languages")
print("  - Kokoro-82M: 82M parameters, high-quality English")
print("  - Both run on laptop CPU without GPU")

print("\nProduction Use Cases:")
print("  1. Voice-over for video ads (attach to 16-frame clips from Ch.9)")
print("  2. Podcast intro generation (custom brand voice)")
print("  3. Accessibility: Alt-text narration for marketing images")

print("\n✅ Chapter 11 Complete: CPU-friendly audio generation")
print("   → No GPU required (unlike image/video generation)")
print("   → Batch generation: Precompute voice-overs for 100 video clips offline")
print("   → Voice cloning: Fine-tune on 5 minutes of client's voice")

## Chapter 12: Generative Evaluation — Measuring What Matters

**Key Concept:** FID measures distribution similarity. CLIP Score measures text-image alignment. LPIPS measures perceptual similarity. HPSv2 predicts human ratings using a model trained on 800k human comparisons.

**What it unlocked:**
- Objective quality measurement: Client surveys said ~3.9/5.0. HPSv2 confirms 4.1/5.0 on 500-image test set
- A/B testing: Compare scheduler performance objectively
- Automated regression testing: Track metric drift after model updates

In [ ]:
# Compute CLIP Score for text-image alignment
def compute_clip_score(image, prompt):
    """Compute CLIP Score: cosine similarity between text and image embeddings"""
    inputs = clip_processor(text=[prompt], images=image, return_tensors="pt", padding=True)
    with torch.no_grad():
        outputs = clip_model(**inputs)
        clip_score = (outputs.logits_per_image / 100.0).item()  # Normalized
    return clip_score

# Evaluate generated image
clip_score = compute_clip_score(generated_image, prompt)

print("Generative Evaluation Metrics:")
print("="*70)
print(f"Prompt: {prompt}")
print(f"CLIP Score: {clip_score:.3f}")
print(f"  → Typical range: >0.28 = good, >0.32 = excellent")
print(f"  → Measures semantic alignment between text prompt and generated image")
print("="*70)

print("\nMetric Selection by Use Case:")
print("-" * 70)
print(f"{'Question':<40} {'Metric':<15} {'Good Range'}")
print("-" * 70)
print(f"{'Does generated set match real data?':<40} {'FID':<15} {'<50'}")
print(f"{'Does image match text prompt?':<40} {'CLIP Score':<15} {'>0.28'}")
print(f"{'Is img2img edit perceptually close?':<40} {'LPIPS':<15} {'<0.3'}")
print(f"{'Will humans prefer this image?':<40} {'HPSv2':<15} {'>3.8'}")
print("-" * 70)

print("\n✅ Chapter 12 Complete: Objective quality metrics for generative AI")
print("   → CLIP Score validates text-image alignment")
print("   → HPSv2 predicts human preference ratings (4.1/5.0 avg for VisualForge)")
print("   → Automated A/B testing: Compare schedulers, guidance scales, models")
print("   → Regression testing: Regenerate 100 fixed prompts after updates")

## Chapter 13: Local Diffusion Lab — Production Assembly & Optimization

**Key Concept:** Assemble all 12 preceding chapters into a production-ready pipeline. Deploy Stable Diffusion locally with SDXL-Turbo (4-step LCM distillation) for 8-second generation.

**What it unlocked:**
- 8-second generation: SDXL-Turbo (4 steps, LCM distillation) hits 8s per 512×512 image
- Production deployment: All components optimized (LCM, xFormers, mixed precision, batch processing)
- Local-first philosophy validated: No cloud GPU required. No API costs. Offline-capable.

In [ ]:
# Production optimization stack
print("Production Optimization Stack (apply in this order):")
print("="*70)
print("1. Scheduler Swap (DDIM 50 steps)       → 30s per image")
print("2. Mixed Precision (fp32 → fp16)        → 15s per image (2× speedup)")
print("3. LCM Distillation (50 → 4 steps)      → 8s per image (2× speedup)")
print("4. xFormers Attention                    → 40% VRAM reduction")
print("5. Batch Processing (10 images)          → 20% per-image speedup")
print("="*70)

# Apply optimizations to existing pipeline
if torch.cuda.is_available():
    print("\nApplying production optimizations...")
    
    # 1. Mixed precision (already applied when loading)
    print("✓ Mixed precision (fp16): Enabled")
    
    # 2. xFormers memory-efficient attention
    try:
        pipe.enable_xformers_memory_efficient_attention()
        print("✓ xFormers attention: Enabled (40% VRAM reduction)")
    except:
        print("⚠ xFormers not available (pip install xformers)")
    
    # 3. DDIM scheduler (already applied)
    print("✓ DDIM scheduler: Enabled (10× faster than DDPM)")
    
    # 4. LCM would require different model (SDXL-Turbo)
    print("⚠ LCM distillation: Requires SDXL-Turbo model (not loaded)")
else:
    print("\nRunning on CPU - optimizations limited:")
    print("✓ DDIM scheduler enabled")
    print("⚠ Mixed precision and xFormers require GPU")

print("\nHardware Recommendations:")
print("-" * 70)
print(f"{'Setup':<20} {'Hardware':<30} {'Speed':<15} {'Cost'}")
print("-" * 70)
print(f"{'Minimum':<20} {'16GB RAM, modern CPU':<30} {'30s/image':<15} {'$1,500'}")
print(f"{'Recommended':<20} {'16GB RAM, RTX 3060':<30} {'12s/image':<15} {'$2,500'}")
print(f"{'Optimal':<20} {'32GB RAM, RTX 4090':<30} {'8s/image':<15} {'$4,000'}")
print("-" * 70)

print("\n✅ Chapter 13 Complete: Production-ready pipeline with optimizations")
print("   → 8-second generation with SDXL-Turbo (4-step LCM)")
print("   → Local-first: $0/month cloud costs vs. $500-2000/month for cloud GPUs")
print("   → All 6 constraints achieved: Quality, Speed, Cost, Control, Throughput, Versatility")

## Production PixelSmith Pipeline — Putting It All Together

Here's the complete end-to-end pipeline that VisualForge uses in production, integrating all 13 chapters.

In [ ]:
def pixelsmith_production_pipeline(brief: dict):
    """
    Complete VisualForge production pipeline integrating all 13 chapters.
    
    Args:
        brief: Campaign brief dict with keys:
            - type: 'product-on-white', 'lifestyle', 'editorial-fashion'
            - prompt: Text description
            - negative: Negative prompt
            - guidance_scale: CFG scale (default 7.5)
            - num_steps: Inference steps (default 20)
            - control_image: Optional reference for ControlNet
            - seed: Random seed for reproducibility
    
    Returns:
        dict with generated image, metrics, and approval status
    """
    import time
    start_time = time.time()
    
    # Step 1: Text encoding (Ch.3 CLIP)
    print(f"\n{'='*70}")
    print(f"VisualForge Production Pipeline - {brief['type']}")
    print(f"{'='*70}")
    print(f"Prompt: {brief['prompt']}")
    print(f"Negative: {brief['negative']}")
    
    # Step 2: Image generation (Ch.6 Latent Diffusion + Ch.7 Guidance)
    generator = torch.Generator(device=device).manual_seed(brief['seed'])
    
    with torch.no_grad():
        result = pipe(
            prompt=brief['prompt'],
            negative_prompt=brief['negative'],
            guidance_scale=brief['guidance_scale'],
            num_inference_steps=brief['num_steps'],
            generator=generator,
            height=512,
            width=512
        )
        generated_image = result.images[0]
    
    generation_time = time.time() - start_time
    
    # Step 3: Quality metrics (Ch.12 Evaluation)
    clip_score = compute_clip_score(generated_image, brief['prompt'])
    
    # Step 4: Auto-QA (Ch.10 Multimodal LLMs)
    # In production, would call LLaVA for visual QA
    # For now, use CLIP score as proxy
    auto_approve = clip_score > 0.28
    
    # Step 5: Decision logic
    if auto_approve:
        status = 'auto_approved'
        print(f"\n✅ AUTO-APPROVED")
    else:
        status = 'human_review'
        print(f"\n⚠️  HUMAN REVIEW REQUIRED")
    
    # Results
    print(f"\nMetrics:")
    print(f"  Generation time: {generation_time:.1f}s")
    print(f"  CLIP Score: {clip_score:.3f}")
    print(f"  Status: {status}")
    print(f"{'='*70}")
    
    return {
        'image': generated_image,
        'status': status,
        'metrics': {
            'clip_score': clip_score,
            'generation_time': generation_time
        },
        'brief': brief
    }

# Example: Run production pipeline
example_brief = {
    'type': 'product-on-white',
    'prompt': 'Premium leather handbag, centered, white background, studio lighting, professional product photography',
    'negative': 'shadows, reflections, text, logos, clutter, background texture, colored background',
    'guidance_scale': 7.5,
    'num_steps': 20,
    'seed': 42
}

result = pixelsmith_production_pipeline(example_brief)

# Display result
plt.figure(figsize=(10, 10))
plt.imshow(result['image'])
plt.axis('off')
plt.title(f"Status: {result['status'].upper()} | CLIP: {result['metrics']['clip_score']:.3f} | Time: {result['metrics']['generation_time']:.1f}s")
plt.tight_layout()
plt.show()

## Final Results — All 6 Constraints Achieved ✅

| Constraint | Target | PixelSmith v6 Result | Chapter(s) |
|------------|--------|---------------------|------------|
| #1 QUALITY | ≥4.0/5.0 | **4.1/5.0** (HPSv2) | Ch.12 Evaluation |
| #2 SPEED | <30s | **8s** (SDXL-Turbo) | Ch.13 LCM Distillation |
| #3 COST | <$5k | **$2,500** laptop | Ch.6 Latent Diffusion |
| #4 CONTROL | <5% unusable | **3%** unusable | Ch.8 ControlNet |
| #5 THROUGHPUT | 100+/day | **120/day** | Ch.10 Auto-QA |
| #6 VERSATILITY | 3 modes | **Text→Image + Video + Understanding** | Ch.9, Ch.10 |

**Business Impact:**
- **$600k/year savings** (eliminated freelancer costs)
- **2.5-month payback** ($125k total investment / $600k annual savings)
- **40× faster turnaround** (5 days → 3 hours for 20-image campaign)
- **8× throughput increase** (15 → 120 images/day capacity)

**Local-First Philosophy:**
- **Privacy:** Client data never leaves laptop
- **Cost:** $0/month vs. $500-2000/month cloud GPUs
- **Latency:** 8 seconds local vs. 15-30 seconds cloud + queue
- **Reliability:** No dependency on third-party API uptime
- **Customization:** Fine-tune on proprietary data without uploading

## Next Steps

1. **Deep dive into individual chapters:** Each section above corresponds to a full chapter with detailed theory, math, and production examples
2. **Load production models:** Replace demo code with full SDXL-Turbo, ControlNet, and LLaVA models
3. **Fine-tune for your use case:** LoRA fine-tuning on brand-specific images (2-3 hours on laptop)
4. **Build your pipeline:** Adapt the `pixelsmith_production_pipeline` function to your workflow
5. **Optimize further:** Add quantization (int8), knowledge distillation, or specialized schedulers

**See Also:**
- [notes/05-multimodal_ai/README.md](README.md) — Track overview and reading paths
- [notes/05-multimodal_ai/grand_solution.md](grand_solution.md) — Complete narrative (this notebook's source)
- [notes/03-ai/README.md](../03-ai/README.md) — LLM fundamentals (prerequisite for Ch.10)
- [AGENTS.md](../../AGENTS.md) — Custom VS Code agents for this repository